# 03 Read And Search

这个 notebook 只做最基础的三件事：

1. 读取 SQLite 里的 embedding
2. 把 `embedding_blob` 还原成 numpy 向量
3. 用最基础的 query embedding 做 top-k 检索，并把命中 chunk 内容打印出来

`BLOB` 的全称是 `Binary Large Object`。这里它存的是 embedding 向量的二进制字节流。

In [1]:
from __future__ import annotations

import sqlite3
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

BASE_DIR = Path('/Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer')
DB_PATH = BASE_DIR / '04_Embedding' / 'embedding_store' / 'filing_embeddings.sqlite3'
MODEL_NAME = 'BAAI/bge-m3'
QUERY_MAX_LENGTH = 512

print('DB_PATH =', DB_PATH)
print('MODEL_NAME =', MODEL_NAME)

DB_PATH = /Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/04_Embedding/embedding_store/filing_embeddings.sqlite3
MODEL_NAME = BAAI/bge-m3


In [2]:
def load_embeddings_from_sqlite(db_path: Path = DB_PATH) -> tuple[list[dict], np.ndarray]:
    conn = sqlite3.connect(db_path)
    rows = conn.execute(
        '''
        SELECT chunk_id, market, company_code, ticker, company_name, report_year,
               filing_date, document_type, title, language, source_path,
               section_name, chunk_index, token_count, char_count, chunk_text,
               embedding_blob, embedding_dim, embedding_dtype
        FROM embeddings
        ORDER BY created_at, rowid
        '''
    ).fetchall()
    progress = conn.execute(
        'SELECT build_name, last_line, processed_chunks, updated_at FROM build_progress ORDER BY updated_at'
    ).fetchall()
    conn.close()

    metadata: list[dict] = []
    vectors: list[np.ndarray] = []
    for row in rows:
        blob = row[16]
        dim = int(row[17])
        dtype = np.dtype(row[18])
        vector = np.frombuffer(blob, dtype=dtype).copy()
        if vector.shape[0] != dim:
            raise ValueError(f'Embedding dim mismatch: expected {dim}, got {vector.shape[0]}')

        metadata.append({
            'chunk_id': row[0],
            'market': row[1],
            'company_code': row[2],
            'ticker': row[3],
            'company_name': row[4],
            'report_year': row[5],
            'filing_date': row[6],
            'document_type': row[7],
            'title': row[8],
            'language': row[9],
            'source_path': row[10],
            'section_name': row[11],
            'chunk_index': row[12],
            'token_count': row[13],
            'char_count': row[14],
            'chunk_text': row[15],
            'embedding_dim': dim,
            'embedding_dtype': str(dtype),
        })
        vectors.append(vector.astype(np.float32))

    matrix = np.vstack(vectors) if vectors else np.empty((0, 0), dtype=np.float32)
    return metadata, matrix, progress


metadata, embedding_matrix, progress_rows = load_embeddings_from_sqlite()
print('embeddings loaded =', len(metadata))
print('embedding matrix shape =', embedding_matrix.shape)
print('build_progress =', progress_rows)
if metadata:
    print('sample chunk_id =', metadata[0]['chunk_id'])
    print('sample vector first 8 values =', embedding_matrix[0][:8])

embeddings loaded = 7984
embedding matrix shape = (7984, 1024)
build_progress = [('bge-m3-priority-sections-v1', 21260, 7984, '2026-03-31 08:58:51')]
sample chunk_id = f89a35396d38335fbb97eeeb8e5483a50a4c32ca
sample vector first 8 values = [-0.02881049  0.03980586 -0.05454496 -0.00890843 -0.03665781 -0.00661147
 -0.03106376  0.03804068]


In [3]:
def pick_device() -> str:
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        return 'mps'
    if torch.cuda.is_available():
        return 'cuda'
    return 'cpu'


class QueryEmbedder:
    def __init__(self, model_name: str = MODEL_NAME):
        self.device = pick_device()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

    def embed_query(self, text: str) -> np.ndarray:
        encoded = self.tokenizer(
            [text],
            padding=True,
            truncation=True,
            max_length=QUERY_MAX_LENGTH,
            return_tensors='pt',
        )
        encoded = {key: value.to(self.device) for key, value in encoded.items()}

        with torch.no_grad():
            output = self.model(**encoded)
            dense = output.last_hidden_state[:, 0]
            dense = torch.nn.functional.normalize(dense, p=2, dim=1)

        return dense.detach().cpu().to(torch.float32).numpy()[0]


query_embedder = QueryEmbedder()
print('query device =', query_embedder.device)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

query device = mps


In [4]:
def search_top_k(query: str, top_k: int = 5):
    if embedding_matrix.size == 0:
        raise ValueError('Database has no embeddings.')

    query_vector = query_embedder.embed_query(query).astype(np.float32)
    scores = embedding_matrix @ query_vector
    order = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(order, start=1):
        item = dict(metadata[idx])
        item['rank'] = rank
        item['score'] = float(scores[idx])
        results.append(item)
    return results


def show_search_results(query: str, top_k: int = 3) -> None:
    results = search_top_k(query, top_k=top_k)
    print(f'QUERY: {query}')
    print(f'TOP_K: {top_k}')
    print('=' * 120)
    for item in results:
        print(f"Rank #{item['rank']} | score={item['score']:.4f}")
        print(f"{item['market']} | {item['company_code']} | {item['company_name']} | {item['report_year']} | {item['document_type']}")
        print(f"section={item['section_name']} | chunk_index={item['chunk_index']} | token_count={item['token_count']}")
        print(f"source={item['source_path']}")
        print('-' * 120)
        print(item['chunk_text'])
        print('=' * 120)


In [5]:
show_search_results('us china tariff has increased', top_k=3)

QUERY: us china tariff has increased
TOP_K: 3
Rank #1 | score=0.6499
US | AAPL | Apple Inc. | 2025 | 10-K
section=item 7 | chunk_index=1 | token_count=417
source=/Users/zhanghongyi/Desktop/26 Spring/Prof Zhao Finance Agent/Report_Crawer/03_Data_Processor/US/parsed_filings/AAPL_10K_2025-10-31.json
------------------------------------------------------------------------------------------------------------------------
Inc. | 2025 Form 10-K | 21 Tariffs and Other Measures Beginning in the second quarter of 2025, new U.S. Tariffs were announced, including additional tariffs on imports from China, India, Japan, South Korea, Taiwan, Vietnam and the EU, among others.

In response, several countries have imposed, or threatened to impose, reciprocal tariffs on imports from the U.S. and other retaliatory measures.

Various modifications to the U.S.

Tariffs have been announced and further changes could be made in the future, which may include additional sector-based tariffs or other measures.

Fo